# Feature engineering — Base 2 — tráfego na I-94

Orquestra `features/temporal.py` com `BASE_2` em `features/regras_bases.py`.

O quadro prevê um passo de `1h`. O horizonte final continua pendente.

## Disponibilidade

- `is_holiday`: **conhecido antecipadamente** (feriado da data-alvo; dicionário aprovado).
- `temp`, `rain_1h`, `snow_1h`, `clouds_all`: observadas em `t` para prever `t+1`.
- `weather_main`, `weather_description` e a coluna textual `holiday`: **proibidas** em `X`.
- Arquivos: `catalogo_features_base2.csv` e `disponibilidade_covariaveis_base2.csv`.


In [1]:
from pathlib import Path
import sys

import pandas as pd

def find_project_root(start):
    for candidate in (start, *start.parents):
        if (candidate / "projeto.yaml").is_file():
            return candidate
    raise FileNotFoundError("Execute a partir do projeto ou de uma subpasta dele.")

ROOT = find_project_root(Path.cwd().resolve())
sys.path.insert(0, str(ROOT / "src"))
from series_temporais.features.regras_bases import (
    BASE_2,
    preparar_entrada_base2,
    validar_proibidas,
)
from series_temporais.features.temporal import (
    build_one_step_frame,
    choque_futuro_nao_altera_features,
    chronological_train_test,
)

BASE_DIR = ROOT / "data" / "base_02"
DATA_PATH = BASE_DIR / "prepared.csv"
CATALOGO_PATH = BASE_DIR / "catalogo_features_base2.csv"
DISPONIBILIDADE_PATH = BASE_DIR / "disponibilidade_covariaveis_base2.csv"
SPEC = BASE_2
TRAIN_RATIO = 0.8


## Quadro de um passo

Cada linha é uma origem. O alvo do modelo é a variação até o passo seguinte; ele não entra nas features.

In [2]:
df = pd.read_csv(DATA_PATH)
df = preparar_entrada_base2(df)
df = df.sort_values(SPEC.time_col, kind="stable").reset_index(drop=True)
validar_proibidas(SPEC.external_cols(), SPEC)
quadro, features = build_one_step_frame(
    df,
    time_col=SPEC.time_col,
    target_col=SPEC.target_col,
    external_cols=SPEC.external_cols(),
    expected_step=SPEC.expected_step,
    lags=SPEC.lags,
    windows=SPEC.windows,
    availability=SPEC.availability(),
)
assert len(quadro) == 16859
assert len(features) == 22
assert "is_holiday_alvo" in features
assert {"y_true", "target_delta", "target_time"}.isdisjoint(features)
quadro[features].head()


,level_t,is_holiday_alvo,temp_t,rain_1h_t,snow_1h_t,clouds_all_t,target_lag_1,target_lag_2,target_lag_3,target_lag_24,...,target_mean_24,target_std_24,target_mean_168,target_std_168,target_hour_sin,target_hour_cos,target_dow_sin,target_dow_cos,target_month_sin,target_month_cos
0,2051.0,0.0,280.10,0.0,0.0,92.0,3077.0,2942.0,2908.0,1177.0,...,3466.000000,1857.524338,3343.833333,2004.142685,0.000000,1.000000,-0.974928,-0.222521,-1.0,-1.836970e-16
1,1270.0,0.0,280.06,0.0,0.0,92.0,2051.0,3077.0,2942.0,699.0,...,3502.416667,1820.489813,3344.750000,2003.515835,0.258819,0.965926,-0.974928,-0.222521,-1.0,-1.836970e-16
2,773.0,0.0,280.28,0.0,0.0,96.0,1270.0,2051.0,3077.0,436.0,...,3526.208333,1787.122337,3344.363095,2003.910143,0.500000,0.866025,-0.974928,-0.222521,-1.0,-1.836970e-16
3,589.0,0.0,280.38,0.0,0.0,96.0,773.0,1270.0,2051.0,353.0,...,3540.250000,1763.960819,3344.386905,2003.879567,0.707107,0.707107,-0.974928,-0.222521,-1.0,-1.836970e-16
4,432.0,0.0,280.62,0.0,0.0,96.0,589.0,773.0,1270.0,399.0,...,3550.083333,1746.739580,3344.500000,2003.723513,0.866025,0.500000,-0.974928,-0.222521,-1.0,-1.836970e-16


## Catálogo

O arquivo CSV fica na pasta do grupo e registra, para cada coluna de `X`, se a informação é conhecida na origem, só no passado ou antecipadamente.

In [3]:
catalogo = SPEC.catalogo(features)
disponibilidade = SPEC.tabela_disponibilidade()
assert not catalogo.grupo.eq("revisar").any()
assert catalogo.set_index("feature").loc["is_holiday_alvo", "status"] == "aprovado"
catalogo.to_csv(CATALOGO_PATH, index=False)
disponibilidade.to_csv(DISPONIBILIDADE_PATH, index=False)
disponibilidade


,coluna,disponibilidade,atraso,feature,status,justificativa
0,is_holiday,conhecido_antecipadamente,0,is_holiday_alvo,aprovado,Feriado da data-alvo; calendário conhecido ant...
1,temp,na_origem,0,temp_t,em_validacao,Observada na hora t; não simula previsão meteo...
2,rain_1h,na_origem,0,rain_1h_t,em_validacao,Acumulado observado na hora t.
3,snow_1h,na_origem,0,snow_1h_t,em_validacao,Acumulado observado na hora t.
4,clouds_all,na_origem,0,clouds_all_t,em_validacao,Cobertura observada na hora t.
5,weather_main,proibida,0,,excluido,Proibida no quadro modelável.
6,weather_description,proibida,0,,excluido,Proibida no quadro modelável.
7,holiday,proibida,0,,excluido,Proibida no quadro modelável.


## Corte cronológico

A última linha de treino cujo alvo cruzaria a primeira origem de teste é removida.

In [4]:
treino, teste = chronological_train_test(quadro, TRAIN_RATIO)
assert len(treino) == 13486
assert len(teste) == 3372
assert treino.target_time.max() < teste.origin_time.min()
pd.DataFrame({
    "recorte": ["treino", "teste"],
    "linhas": [len(treino), len(teste)],
    "primeira_origem": [treino.origin_time.min(), teste.origin_time.min()],
    "ultima_origem": [treino.origin_time.max(), teste.origin_time.max()],
})


,recorte,linhas,primeira_origem,ultima_origem
0,treino,13486,2012-10-19 23:00:00,2018-04-14 23:00:00
1,teste,3372,2018-04-15 01:00:00,2018-09-30 22:00:00


## Checagem de leakage

A segunda metade da série recebe um valor artificial. As features das origens anteriores precisam permanecer idênticas.

In [5]:
# is_holiday é calendário da data-alvo: alterar o "futuro" da série não simula
# vazamento observacional; o choque cobre só variáveis observadas.
resultado = choque_futuro_nao_altera_features(
    df,
    time_col=SPEC.time_col,
    indice_inicial_do_choque=len(df) // 2,
    colunas=[SPEC.target_col, "temp", "rain_1h", "snow_1h", "clouds_all"],
    target_col=SPEC.target_col,
    external_cols=SPEC.external_cols(),
    expected_step=SPEC.expected_step,
    lags=SPEC.lags,
    windows=SPEC.windows,
    availability=SPEC.availability(),
)
assert resultado["iguais"] is True
assert resultado["diferenca_maxima"] == 0
resultado


{'origens_comparadas': 2873, 'diferenca_maxima': 0.0, 'iguais': True}

## Onde ficou

- Código: `src/series_temporais/features/temporal.py`
- Catálogo gerado por este notebook: `data/base_02/catalogo_features_base2.csv`
- O quadro completo permanece em memória. O notebook de Random Forest do grupo chama a mesma função e o mesmo conjunto de features.
